# SuperADD — exact upstream reproduction + AU-PRO\(_{0.05}\) + MVTec AD 2 submission

Notebook này **không sửa `model.py`, không đổi backbone, không đổi post-processing, không tự tối ưu threshold**.

- Upstream: `LukasRoom/SuperADD`
- Pin commit: `44cf25144442fbbc1334ea59d1632327a4376d1a`
- Public benchmark để đối chiếu paper: **AU-ROC\(_{0.05}\)** + **Segmentation F1** do chính `test-industrial` của SuperADD tính.
- Metric bổ sung cho nghiên cứu: **AU-PRO\(_{0.05}\)** trên toàn bộ `test_public` (good + bad).
- Submission MVTec AD 2: chạy `test_private` + `test_private_mixed`, xuất **float16 TIFF** và **binary PNG**, rồi dùng chính `submit-industrial`.

> Quan trọng: paper SuperADD báo **AU-ROC\(_{0.05}\)**, không phải AU-PRO\(_{0.05}\). Vì vậy không được so AU-PRO với cột AU-ROC của paper.

## 1. Mount Drive và khai báo đường dẫn

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
from pathlib import PurePosixPath

PROJECT_DRIVE = Path("/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9")

ARCHIVE_DIR = PROJECT_DRIVE / "data"
DINO_WEIGHT_DIR = PROJECT_DRIVE / "weights"
PRETRAINED_CACHE = PROJECT_DRIVE / "superadd_pretrained" / "models"
DRIVE_OUT = PROJECT_DRIVE / "superadd_results_exact"

REPO = Path("/content/SuperADD")
DATA_ROOT = REPO / "data" / "mvtec_ad_2"

COMMIT = "44cf25144442fbbc1334ea59d1632327a4376d1a"

CATEGORIES = [
    #"can",
    #"fabric",
    #"fruit_jelly",
    "rice",
    #"sheet_metal", "vial", "wallplugs", "walnuts"
]

PUBLIC_SPLIT = ["test_public"]
PRIVATE_SPLITS = ["test_private", "test_private_mixed"]

assert ARCHIVE_DIR.is_dir(), ARCHIVE_DIR
assert DINO_WEIGHT_DIR.is_dir(), DINO_WEIGHT_DIR

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
PRETRAINED_CACHE.mkdir(parents=True, exist_ok=True)

print("Project :", PROJECT_DRIVE)
print("Archives:", ARCHIVE_DIR)
print("Weights :", DINO_WEIGHT_DIR)
print("Output  :", DRIVE_OUT)

Mounted at /content/drive
Project : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9
Archives: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/data
Weights : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/weights
Output  : /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/superadd_results_exact


## 2. Clone đúng commit và cài đúng workspace bằng `uv`

In [ ]:
def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)

run(["nvidia-smi"])

if REPO.exists():
    shutil.rmtree(REPO)

run(["git", "clone", "https://github.com/LukasRoom/SuperADD.git", REPO])
run(["git", "checkout", "--detach", COMMIT], cwd=REPO)

actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == COMMIT, (actual, COMMIT)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv")
assert UV is not None

run([UV, "sync", "--locked", "--all-packages", "--python", "3.11"], cwd=REPO)

dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
assert dirty == "", f"Source vừa clone đã bị thay đổi:\n{dirty}"

print("SUPERADD SOURCE: PASS", actual)

$ nvidia-smi
$ git clone https://github.com/LukasRoom/SuperADD.git /content/SuperADD
$ git checkout --detach 44cf25144442fbbc1334ea59d1632327a4376d1a
$ /usr/bin/python3 -m pip install -q uv
$ /usr/local/bin/uv sync --locked --all-packages --python 3.11
SUPERADD SOURCE: PASS 44cf25144442fbbc1334ea59d1632327a4376d1a


## 3. Giải nén đúng 3 split cần dùng

In [ ]:
SPLITS_NEEDED = {"test_public", "test_private", "test_private_mixed"}

def extract_selected_splits(archive: Path, destination: Path, splits=SPLITS_NEEDED):
    destination.mkdir(parents=True, exist_ok=True)

    for split in splits:
        shutil.rmtree(destination / split, ignore_errors=True)

    copied = {s: 0 for s in splits}

    with tarfile.open(archive, "r:gz") as tf:
        for member in tf.getmembers():
            parts = PurePosixPath(member.name).parts
            split_positions = [i for i, p in enumerate(parts) if p in splits]
            if not split_positions:
                continue

            i = split_positions[0]
            split = parts[i]
            rel_parts = parts[i:]

            if any(p in ("", ".", "..") for p in rel_parts):
                raise RuntimeError(f"Unsafe path: {member.name}")

            target = destination.joinpath(*rel_parts)

            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue

            if not member.isfile():
                raise RuntimeError(f"Unsupported archive entry: {member.name}")

            target.parent.mkdir(parents=True, exist_ok=True)
            src = tf.extractfile(member)
            if src is None:
                raise RuntimeError(member.name)

            with src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)

            copied[split] += 1

    for split in splits:
        assert (destination / split).exists(), f"Thiếu {destination/split}"
        assert copied[split] > 0, f"Không extract được file cho {split} từ {archive.name}"

    return copied

archives = {p.name[:-7]: p for p in ARCHIVE_DIR.glob("*.tar.gz")}
missing = set(CATEGORIES) - set(archives)
assert not missing, f"Thiếu archive: {sorted(missing)}"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

for category in CATEGORIES:
    print("\n==", category, "==")
    counts = extract_selected_splits(archives[category], DATA_ROOT / category)
    print(counts)

for c in CATEGORIES:
    pub = DATA_ROOT / c / "test_public"
    assert list((pub / "bad").glob("*.png")), f"{c}: thiếu public/bad"
    assert list((pub / "good").glob("*.png")), f"{c}: thiếu public/good"
    assert list((pub / "ground_truth" / "bad").glob("*_mask.png")), f"{c}: thiếu GT"
    assert list((DATA_ROOT / c / "test_private").glob("*.png")), f"{c}: thiếu private"
    assert list((DATA_ROOT / c / "test_private_mixed").glob("*.png")), f"{c}: thiếu private_mixed"

print("\nDATASET: PASS", DATA_ROOT)


== rice ==
{'test_public': 222, 'test_private_mixed': 277, 'test_private': 277}

DATASET: PASS /content/SuperADD/data/mvtec_ad_2


## 4. DINOv3 checkpoint + pretrained SuperADD memory banks

In [ ]:
import requests
from tqdm.auto import tqdm
import numpy as np

weight_candidates = sorted(DINO_WEIGHT_DIR.glob("dinov3_vith16plus_pretrain_*.pth"))
assert len(weight_candidates) == 1, (
    "Cần đúng 1 checkpoint dinov3_vith16plus_pretrain_*.pth trong "
    f"{DINO_WEIGHT_DIR}; hiện có {weight_candidates}"
)

(REPO / "weights").mkdir(exist_ok=True)

weight_src = weight_candidates[0]
weight_dst = REPO / "weights" / weight_src.name

if not weight_dst.exists() or weight_dst.stat().st_size != weight_src.stat().st_size:
    shutil.copy2(weight_src, weight_dst)

assert weight_dst.stat().st_size > 100_000_000, "Checkpoint quá nhỏ / sai file."
print("DINO:", weight_dst.name, f"{weight_dst.stat().st_size/2**30:.2f} GiB")


PUBLIC_SHARE = "https://owncloud.fraunhofer.de/index.php/s/THkX7W8AhRd2RCs/download"

def download_pretrained_file(filename: str, dst: Path):
    if dst.exists() and dst.stat().st_size > 0:
        return

    tmp = dst.with_suffix(dst.suffix + ".part")
    tmp.unlink(missing_ok=True)

    with requests.get(
        PUBLIC_SHARE,
        params={"path": "/", "files": filename},
        stream=True,
        timeout=(30, 300),
    ) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))

        with open(tmp, "wb") as f, tqdm(
            total=total or None, unit="B", unit_scale=True, desc=filename
        ) as bar:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    assert tmp.stat().st_size > 0
    tmp.replace(dst)

def validate_model_pair(category: str, folder: Path):
    jp = folder / f"{category}.json"
    npz = folder / f"{category}.npz"

    meta = json.loads(jp.read_text())
    assert "constructor" in meta and "trained" in meta

    ctor = meta["constructor"]
    assert ctor["backbone"] == "dinov3_vith16plus"
    assert ctor["layers"] == [7, 15, 23, 31]
    assert int(ctor["patch_size"]) == 640
    assert int(ctor["patch_overlap"]) == 128

    z = np.load(npz)
    assert set(z.files) == {"7", "15", "23", "31"}, z.files
    z.close()

    return float(meta["trained"]["threshold"])

for category in CATEGORIES:
    for suffix in (".json", ".npz"):
        p = PRETRAINED_CACHE / f"{category}{suffix}"
        download_pretrained_file(p.name, p)

    thr = validate_model_pair(category, PRETRAINED_CACHE)
    print(category, "threshold =", thr)

(REPO / "models").mkdir(exist_ok=True)

for category in CATEGORIES:
    for suffix in (".json", ".npz"):
        src = PRETRAINED_CACHE / f"{category}{suffix}"
        dst = REPO / "models" / src.name

        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)

print("PRETRAINED MODELS: PASS")

DINO: dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth 3.13 GiB
rice threshold = 0.8007440567016602
PRETRAINED MODELS: PASS


## 5. Hàm ghi config

Chỉ thay **đường dẫn, split, category, cờ lưu prediction**. Các tham số model khi load pretrained được đọc từ `models/<category>.json`.

Để khớp `test.py` của SuperADD và Table 1 paper, public evaluation dùng `evaluate_good_images = false`.

In [ ]:
UPSTREAM_CONFIG = json.loads(
    subprocess.check_output(
        ["git", "show", f"{COMMIT}:config.json"],
        cwd=REPO,
        text=True
    )
)

def write_config(results_dir: str, test_split):
    cfg = dict(UPSTREAM_CONFIG)

    cfg.update({
        "datasets_dir": "./data/",
        "dino_weights_dir": "./weights/",
        "models_dir": "./models/",
        "results_dir": results_dir,
        "submissions_dir": "./submissions/",
        "categories": CATEGORIES,
        "test_split": list(test_split),
        "save_predictions": True,
        "evaluate_good_images": False,
    })

    (REPO / "config.json").write_text(json.dumps(cfg, indent=4))
    return cfg

cfg = write_config("./outputs_public/", PUBLIC_SPLIT)
print(json.dumps(cfg, indent=2))

{
  "datasets_dir": "./data/",
  "dino_weights_dir": "./weights/",
  "models_dir": "./models/",
  "results_dir": "./outputs_public/",
  "submissions_dir": "./submissions/",
  "categories": [
    "rice"
  ],
  "train_fraction": 1,
  "threshold_fraction": 8,
  "brightness_augmentation": [
    0.8,
    1.2
  ],
  "patch_size": 640,
  "patch_overlap": 128,
  "backbone": "dinov3_vith16plus",
  "layers": [
    7,
    15,
    23,
    31
  ],
  "max_database_size": 100000,
  "subsampling_iterations": 100,
  "threshold_percentile": 95,
  "threshold_factor": 1.421,
  "evaluation_downscale": 4,
  "closing_radius": 26,
  "closing_angles": 16,
  "closing_lower_threshold": 0.8,
  "binary_erosion": 1,
  "test_split": [
    "test_public"
  ],
  "save_predictions": true,
  "evaluate_good_images": false
}


## 6. Public — chạy **đúng `test-industrial` của SuperADD**

Kết quả dùng để đối chiếu paper:
- cột `AU` = pixel-level standardized partial **AU-ROC\(_{0.05}\)**
- cột `F1` = F1 của **binary map do threshold/post-processing SuperADD sinh ra**

In [ ]:
PUBLIC_OUT = REPO / "outputs_public"
shutil.rmtree(PUBLIC_OUT, ignore_errors=True)
PUBLIC_OUT.mkdir(parents=True, exist_ok=True)

import pandas as pd
import subprocess, os

def run_streamed(cmd, cwd=None, env=None, tail_n=100):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd), flush=True)

    merged_env = os.environ.copy()
    merged_env["MPLBACKEND"] = "Agg"

    if env:
        merged_env.update(env)

    merged_env["PYTHONUNBUFFERED"] = "1"

    p = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    assert p.stdout is not None

    for line in p.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail) > tail_n:
            tail.pop(0)

    rc = p.wait()
    if rc != 0:
        raise RuntimeError(
            f"\\n[FAILED returncode={rc}]\\n" + "".join(tail)
        )
    return rc

category_rows = []

for category in CATEGORIES:
    print("\\n" + "=" * 80)
    print("PUBLIC CATEGORY:", category)
    print("=" * 80)

    cfg = write_config("./outputs_public/", ["test_public"])
    cfg["categories"] = [category]
    cfg["evaluate_good_images"] = False
    cfg["save_predictions"] = True
    (REPO / "config.json").write_text(json.dumps(cfg, indent=4))

    score_csv = PUBLIC_OUT / "scores_test_public.csv"
    score_csv.unlink(missing_ok=True)

    try:
        run_streamed(
            [UV, "run", "--no-sync", "test-industrial"],
            cwd=REPO,
        )
    except Exception:
        print("\\nFAILED CATEGORY =", category)
        print("GPU state:")
        subprocess.run(["nvidia-smi"])
        raise

    assert score_csv.is_file(), f"Không tạo được {score_csv}"

    one = pd.read_csv(score_csv)
    one = one.loc[:, ~one.columns.str.startswith("Unnamed")]

    row = one[one["Category"] == category]
    assert len(row) == 1, one
    category_rows.append(row.iloc[0].to_dict())

scores = pd.DataFrame(category_rows, columns=["Category", "AU", "F1"])
scores.loc[len(scores)] = [
    "mean",
    scores["AU"].mean(),
    scores["F1"].mean(),
]
scores.to_csv(PUBLIC_OUT / "scores_test_public.csv", index=False)

paper = pd.DataFrame({
    "Category": [
        "can", "fabric", "fruit_jelly", "rice",
        "sheet_metal", "vial", "wallplugs", "walnuts", "mean"
    ],
    "Paper_AUROC005_pct": [
        51.61, 84.20, 81.19, 96.11,
        93.18, 82.59, 92.53, 90.03, 83.93
    ],
    "Paper_F1_pct": [
        0.00, 93.74, 54.68, 73.31,
        59.54, 64.77, 79.16, 75.69, 62.61
    ],
})

report = scores.copy()
report["AUROC005_pct"] = report["AU"] * 100
report["F1_pct"] = report["F1"] * 100
report = report.merge(paper, on="Category", how="left")
report["d_AUROC005_pp"] = (
    report["AUROC005_pct"] - report["Paper_AUROC005_pct"]
)
report["d_F1_pp"] = (
    report["F1_pct"] - report["Paper_F1_pct"]
)

display(report[[
    "Category",
    "AUROC005_pct", "Paper_AUROC005_pct", "d_AUROC005_pp",
    "F1_pct", "Paper_F1_pct", "d_F1_pp",
]])

report.to_csv(
    PUBLIC_OUT / "paper_comparison_test_public.csv",
    index=False,
)

print("\\nPUBLIC TEST FINISHED.")

\n================================================================================
PUBLIC CATEGORY: rice
$ /usr/local/bin/uv run --no-sync test-industrial

processing split test_public
Downloading: "https://github.com/facebookresearch/dinov3/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "file:///content/SuperADD/weights/dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth

  0%|          | 0.00/3.13G [00:00<?, ?B/s]
  1%|          | 19.1M/3.13G [00:00<00:16, 201MB/s]
  1%|          | 39.8M/3.13G [00:00<00:15, 210MB/s]
  2%|▏         | 61.8M/3.13G [00:00<00:15, 219MB/s]
  3%|▎         | 82.8M/3.13G [00:00<00:17, 185MB/s]
  3%|▎         | 105M/3.13G [00:00<00:16, 202MB/s] 
  4%|▍         | 125M/3.13G [00:00<00:18, 175MB/s]
  5%|▍         | 147M/3.13G [00:00<00:16, 191MB/s]
  5%|▌         | 173M/3.13G [00:00<00:14, 214MB/s]
  6%|▌         | 194M/3.13G [00:01<00:15, 209MB/s]
  7%|▋         |

,Category,AUROC005_pct,Paper_AUROC005_pct,d_AUROC005_pp,F1_pct,Paper_F1_pct,d_F1_pp
0,rice,96.112538,96.11,0.002538,73.313419,73.31,0.003419
1,mean,96.112538,83.93,12.182538,73.313419,62.61,10.703419


\nPUBLIC TEST FINISHED.


## 7. Metric bổ sung: AU-PRO\(_{0.05}\) trên **toàn bộ `test_public`**

Paper SuperADD **không báo metric này**. Cell này giữ nguyên `model.predict()` của SuperADD và:
- dùng TIFF float16 đã sinh cho ảnh `bad`;
- chỉ inference thêm ảnh `good` để tính FPR trên toàn bộ pixel normal;
- resize GT về resolution anomaly map bằng nearest-neighbor;
- mỗi connected component GT có trọng số bằng nhau;
- tích phân PRO–FPR từ 0 đến 0.05 và chuẩn hóa chia 0.05.

In [ ]:
# ============================================================
# CELL 7 — AU-PRO@0.05 trên test_public
# SuperADD prediction giữ nguyên, chỉ bổ sung metric AU-PRO
# ============================================================

from pathlib import Path
import os
import subprocess
import textwrap
import pandas as pd

# Chỉ chạy 1 category
AU_PRO_CATEGORIES = ["rice"]

PUBLIC_OUT = REPO / "outputs_public"

AU_PRO_SCRIPT = r"""
from pathlib import Path

import numpy as np
import cv2
import tifffile
import pandas as pd

from torchvision import transforms
from tqdm import tqdm

from industrial.model import SuperADD
from industrial.dataset import MVTecAD2Dataset
from industrial.paths import get_dataset_path, get_model_path


# ============================================================
# CONFIG
# ============================================================

CATEGORIES = ["rice"]

PUBLIC_OUT = Path("./outputs_public")

MAX_FPR = 0.05


# ============================================================
# AU-PRO@0.05
# ============================================================

def compute_aupro_005(normal_scores, region_scores):

    normal_scores = np.concatenate(
        normal_scores
    ).astype(
        np.float32,
        copy=False
    )

    num_regions = len(region_scores)

    if num_regions == 0:
        raise RuntimeError(
            "Không tìm thấy GT connected component."
        )

    anomaly_scores = []
    anomaly_weights = []

    # Mỗi GT region có trọng số bằng nhau
    for scores in region_scores:

        scores = np.asarray(
            scores,
            dtype=np.float32
        ).reshape(-1)

        if scores.size == 0:
            continue

        anomaly_scores.append(scores)

        anomaly_weights.append(
            np.full(
                scores.size,
                1.0 / (num_regions * scores.size),
                dtype=np.float64
            )
        )

    anomaly_scores = np.concatenate(
        anomaly_scores
    )

    anomaly_weights = np.concatenate(
        anomaly_weights
    )

    num_normal_pixels = normal_scores.size

    if num_normal_pixels == 0:
        raise RuntimeError(
            "Không có normal pixels để tính FPR."
        )

    # Score của toàn bộ pixel
    scores_all = np.concatenate([
        normal_scores,
        anomaly_scores
    ])

    # Normal pixel đóng góp vào FPR
    fpr_delta = np.concatenate([
        np.full(
            num_normal_pixels,
            1.0 / num_normal_pixels,
            dtype=np.float64
        ),
        np.zeros(
            anomaly_scores.size,
            dtype=np.float64
        )
    ])

    # Anomaly pixel đóng góp vào PRO
    pro_delta = np.concatenate([
        np.zeros(
            num_normal_pixels,
            dtype=np.float64
        ),
        anomaly_weights
    ])

    # Threshold sweep từ score lớn -> nhỏ
    order = np.argsort(
        scores_all,
        kind="mergesort"
    )[::-1]

    sorted_scores = scores_all[order]

    cumulative_fpr = np.cumsum(
        fpr_delta[order]
    )

    cumulative_pro = np.cumsum(
        pro_delta[order]
    )

    # Chỉ lấy cuối mỗi nhóm có cùng anomaly score
    group_ends = np.r_[
        np.flatnonzero(
            sorted_scores[1:]
            !=
            sorted_scores[:-1]
        ),
        sorted_scores.size - 1
    ]

    fpr = np.r_[
        0.0,
        cumulative_fpr[group_ends]
    ]

    pro = np.r_[
        0.0,
        cumulative_pro[group_ends]
    ]

    # Loại FPR trùng
    unique_fpr, first_idx, counts = np.unique(
        fpr,
        return_index=True,
        return_counts=True
    )

    unique_pro = pro[
        first_idx + counts - 1
    ]

    if unique_fpr[-1] < MAX_FPR:
        raise RuntimeError(
            "FPR curve không đạt 0.05."
        )

    # ========================================================
    # Cắt curve chính xác tại FPR = 0.05
    # ========================================================

    idx = np.searchsorted(
        unique_fpr,
        MAX_FPR,
        side="left"
    )

    if (
        idx < len(unique_fpr)
        and
        np.isclose(
            unique_fpr[idx],
            MAX_FPR
        )
    ):

        x = unique_fpr[:idx + 1]
        y = unique_pro[:idx + 1]

    else:

        x0 = unique_fpr[idx - 1]
        x1 = unique_fpr[idx]

        y0 = unique_pro[idx - 1]
        y1 = unique_pro[idx]

        y_at_max_fpr = (
            y0
            +
            (MAX_FPR - x0)
            *
            (y1 - y0)
            /
            (x1 - x0)
        )

        x = np.r_[
            unique_fpr[:idx],
            MAX_FPR
        ]

        y = np.r_[
            unique_pro[:idx],
            y_at_max_fpr
        ]

    # Chuẩn hóa diện tích về [0,1]
    area = np.trapezoid(
        y,
        x
    )

    return float(
        area / MAX_FPR
    )


# ============================================================
# EVALUATION
# ============================================================

rows = []

for category in CATEGORIES:

    print()
    print("=" * 70)
    print(
        f"AU-PRO@0.05 — {category}"
    )
    print("=" * 70)

    dataset = MVTecAD2Dataset(
        get_dataset_path(
            "mvtec_ad_2"
        ),
        category,
        "test_public",
        transform=transforms.ToTensor()
    )

    # Bad images đã được Cell 6 inference và lưu TIFF
    bad_prediction_dir = (
        PUBLIC_OUT
        /
        "anomaly_images"
        /
        category
        /
        "test_public"
    )

    model = None

    normal_scores = []
    region_scores = []

    num_good = 0
    num_bad = 0

    for sample in tqdm(
        dataset,
        desc=category
    ):

        stem = Path(
            sample.image_path
        ).stem

        # ====================================================
        # BAD IMAGE
        # Dùng lại prediction TIFF từ Cell 6
        # ====================================================

        if sample.label == 1:

            prediction_path = (
                bad_prediction_dir
                /
                f"{stem}.tiff"
            )

            if not prediction_path.is_file():

                raise FileNotFoundError(
                    f"Thiếu prediction: "
                    f"{prediction_path}"
                )

            anomaly_map = np.asarray(
                tifffile.imread(
                    prediction_path
                ),
                dtype=np.float16
            ).astype(
                np.float32
            )

            num_bad += 1

        # ====================================================
        # GOOD IMAGE
        # Cell 6 không inference good vì
        # evaluate_good_images=False.
        # Cần predict để tính normal-pixel FPR cho AU-PRO.
        # ====================================================

        else:

            if model is None:

                model = SuperADD.from_disk(
                    get_model_path(
                        category
                    )
                )

            anomaly_map, _ = model.predict(
                sample.image
            )

            # Đồng nhất precision với TIFF do test.py lưu
            anomaly_map = (
                anomaly_map
                .astype(
                    np.float16
                )
                .astype(
                    np.float32
                )
            )

            num_good += 1

        # ====================================================
        # GT MASK
        # ====================================================

        if sample.mask is None:

            gt = np.zeros(
                anomaly_map.shape,
                dtype=np.uint8
            )

        else:

            gt_original = (
                sample.mask[0]
                .cpu()
                .numpy()
                >
                0.5
            ).astype(
                np.uint8
            )

            # Resize mask -> đúng resolution anomaly map
            gt = cv2.resize(
                gt_original,
                anomaly_map.shape[::-1],
                interpolation=cv2.INTER_NEAREST
            )

            gt = (
                gt > 0
            ).astype(
                np.uint8
            )

        # ====================================================
        # NORMAL PIXELS
        # ====================================================

        normal_scores.append(
            anomaly_map[
                gt == 0
            ].reshape(-1)
        )

        # ====================================================
        # GT CONNECTED COMPONENTS
        # ====================================================

        if gt.any():

            num_labels, labels = (
                cv2.connectedComponents(
                    gt,
                    connectivity=8
                )
            )

            for region_id in range(
                1,
                num_labels
            ):

                region_scores.append(
                    anomaly_map[
                        labels
                        ==
                        region_id
                    ].reshape(-1)
                )

    # ========================================================
    # CALCULATE
    # ========================================================

    au_pro = compute_aupro_005(
        normal_scores,
        region_scores
    )

    rows.append({
        "Category": category,
        "AU_PRO_0.05": au_pro,
        "AU_PRO_0.05_pct": au_pro * 100.0,
        "good_images": num_good,
        "bad_images": num_bad,
        "gt_regions": len(region_scores)
    })

    print()
    print(
        f"{category}: "
        f"AU-PRO@0.05 = "
        f"{au_pro * 100:.2f}%"
    )


# ============================================================
# SAVE RESULT
# ============================================================

df = pd.DataFrame(
    rows
)

if len(df) > 1:

    mean_row = {
        "Category": "mean",
        "AU_PRO_0.05":
            df["AU_PRO_0.05"].mean(),
        "AU_PRO_0.05_pct":
            df["AU_PRO_0.05_pct"].mean(),
        "good_images":
            df["good_images"].sum(),
        "bad_images":
            df["bad_images"].sum(),
        "gt_regions":
            df["gt_regions"].sum()
    }

    df.loc[len(df)] = mean_row


output_path = (
    PUBLIC_OUT
    /
    "aupro_005_test_public.csv"
)

df.to_csv(
    output_path,
    index=False
)

print()
print("=" * 70)
print(df.to_string(index=False))
print("=" * 70)
print()
print(
    "Saved:",
    output_path
)
"""


# ============================================================
# GHI SCRIPT RA FILE
# Tránh lỗi khi truyền một script rất dài qua python -c
# ============================================================

script_path = (
    REPO
    /
    "_compute_aupro_005.py"
)

script_path.write_text(
    AU_PRO_SCRIPT
)


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["PYTHONUNBUFFERED"] = "1"

# Fix lỗi matplotlib backend trên Colab
env["MPLBACKEND"] = "Agg"


# ============================================================
# RUN TRONG UV ENV CỦA SUPERADD
# ============================================================

cmd = [
    str(UV),
    "run",
    "--no-sync",
    "python",
    "-u",
    str(script_path)
]

print(
    "$",
    " ".join(cmd)
)


process = subprocess.Popen(
    cmd,
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

logs = []

for line in process.stdout:

    print(
        line,
        end="",
        flush=True
    )

    logs.append(
        line
    )


return_code = process.wait()


if return_code != 0:

    raise RuntimeError(
        "\nAU-PRO evaluation FAILED:\n"
        +
        "".join(
            logs[-100:]
        )
    )


# ============================================================
# DISPLAY RESULT
# ============================================================

result_csv = (
    PUBLIC_OUT
    /
    "aupro_005_test_public.csv"
)

assert result_csv.is_file(), (
    f"Không tìm thấy {result_csv}"
)

result_df = pd.read_csv(
    result_csv
)

display(
    result_df
)

$ /usr/local/bin/uv run --no-sync python -u /content/SuperADD/_compute_aupro_005.py

AU-PRO@0.05 — rice

rice:   0%|          | 0/132 [00:00<?, ?it/s]Using cache found in /root/.cache/torch/hub/facebookresearch_dinov3_main

rice: 100%|██████████| 132/132 [11:49<00:00,  5.37s/it]

rice: AU-PRO@0.05 = 71.61%

Category  AU_PRO_0.05  AU_PRO_0.05_pct  good_images  bad_images  gt_regions
    rice     0.716051        71.605128           42          90         120

Saved: outputs_public/aupro_005_test_public.csv


,Category,AU_PRO_0.05,AU_PRO_0.05_pct,good_images,bad_images,gt_regions
0,rice,0.716051,71.605128,42,90,120


## 8. Private + private-mixed — tạo đúng submission của SuperADD

`submit-industrial` yêu cầu đủ 8 category và cả hai split private.

Nó kiểm tra:
- TIFF: 2D, `float16`
- PNG: 2D, chỉ `{0,255}`
- tên: `###_regular` cho private và `###_mixed` cho private-mixed
- đúng số lượng từng category.

In [ ]:
# ============================================================
# CELL 8 — PRIVATE + PRIVATE_MIXED INFERENCE / SUBMISSION
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

CATEGORIES = ["fruit_jelly"]

PRIVATE_SPLITS = [
    "test_private",
    "test_private_mixed",
]

SUBMIT_OUT = REPO / "outputs"
SUBMISSIONS = REPO / "submissions"


# ------------------------------------------------------------
# CLEAN OUTPUT CŨ
# CHỈ chạy khi bắt đầu inference lại từ đầu
# ------------------------------------------------------------

shutil.rmtree(
    SUBMIT_OUT,
    ignore_errors=True
)

shutil.rmtree(
    SUBMISSIONS,
    ignore_errors=True
)


# ------------------------------------------------------------
# WRITE CONFIG
# ------------------------------------------------------------

cfg = write_config(
    "./outputs/",
    PRIVATE_SPLITS
)

# Ép chắc chắn chỉ chạy category đã chọn
cfg["categories"] = CATEGORIES
cfg["test_split"] = PRIVATE_SPLITS
cfg["save_predictions"] = True
cfg["evaluate_good_images"] = False

(REPO / "config.json").write_text(
    json.dumps(
        cfg,
        indent=4
    )
)

print("Categories :", CATEGORIES)
print("Splits     :", PRIVATE_SPLITS)


# ------------------------------------------------------------
# ENVIRONMENT
# ------------------------------------------------------------

env = os.environ.copy()

env["PYTHONUNBUFFERED"] = "1"

# Fix matplotlib backend trên Colab
env["MPLBACKEND"] = "Agg"


# ------------------------------------------------------------
# STREAMED RUNNER
# ------------------------------------------------------------

def run_streamed(
    cmd,
    cwd=None,
    env=None,
    tail_n=120
):

    cmd = [
        str(x)
        for x in cmd
    ]

    print(
        "\n$",
        " ".join(cmd),
        flush=True
    )

    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    logs = []

    for line in process.stdout:

        print(
            line,
            end="",
            flush=True
        )

        logs.append(line)

        if len(logs) > tail_n:
            logs.pop(0)

    return_code = process.wait()

    if return_code != 0:

        raise RuntimeError(
            "\n"
            f"PROCESS FAILED "
            f"(return code={return_code})\n"
            +
            "".join(logs)
        )


# ------------------------------------------------------------
# INFERENCE
# ------------------------------------------------------------

run_streamed(
    [
        UV,
        "run",
        "--no-sync",
        "test-industrial"
    ],
    cwd=REPO,
    env=env
)


# ------------------------------------------------------------
# KIỂM TRA OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERENCE FINISHED")
print("=" * 70)

for category in CATEGORIES:

    for split in PRIVATE_SPLITS:

        folder = (
            SUBMIT_OUT
            / "anomaly_images"
            / category
            / split
        )

        files = list(
            folder.glob("*.tiff")
        )

        print(
            f"{category:12s} | "
            f"{split:20s} | "
            f"{len(files)} TIFF"
        )


# ------------------------------------------------------------
# SUBMISSION
# Chỉ được tạo khi đủ cả 8 category
# ------------------------------------------------------------

ALL_CATEGORIES = [
    "can",
    "fabric",
    "fruit_jelly",
    "rice",
    "sheet_metal",
    "vial",
    "wallplugs",
    "walnuts",
]

if set(CATEGORIES) == set(ALL_CATEGORIES):

    print(
        "\nĐủ 8 category -> tạo submission..."
    )

    run_streamed(
        [
            UV,
            "run",
            "--no-sync",
            "submit-industrial"
        ],
        cwd=REPO,
        env=env
    )

    archives = sorted(
        SUBMISSIONS.glob(
            "submission_*.tar.gz"
        ),
        key=lambda p:
            p.stat().st_mtime
    )

    assert archives, (
        "Không tạo được submission."
    )

    submission = archives[-1]

    print(
        "\nSUBMISSION:",
        submission
    )

    print(
        "SIZE GiB:",
        submission.stat().st_size
        / 2**30
    )

else:

    print(
        "\nChỉ inference:",
        CATEGORIES
    )

    print(
        "Chưa tạo submission vì "
        "chưa đủ 8 category."
    )

## 9. Kiểm tra cuối + lưu về Drive

In [ ]:
import tifffile
from PIL import Image
import numpy as np
from datetime import datetime, timezone

sample_tiff = next(
    (SUBMIT_OUT / "anomaly_images" / "fruit_jelly" / "test_private")
    .glob("*.tiff")
)

sample_png = next(
    (SUBMIT_OUT / "anomaly_images_thresholded" / "fruit_jelly" / "test_private")
    .glob("*.png")
)

a = np.asarray(tifffile.imread(sample_tiff))
b = np.asarray(Image.open(sample_png))

assert a.ndim == 2 and a.dtype == np.float16
assert b.ndim == 2
assert set(np.unique(b).tolist()).issubset({0, 255})

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
save_dir = DRIVE_OUT / run_id
save_dir.mkdir(parents=True, exist_ok=True)

for p in [
    PUBLIC_OUT / "scores_test_public.csv",
    PUBLIC_OUT / "paper_comparison_test_public.csv",
    PUBLIC_OUT / "aupro_005_test_public.csv",
]:
    assert p.is_file(), p
    shutil.copy2(p, save_dir / p.name)

submission_dst = save_dir / submission.name
shutil.copy2(submission, submission_dst)

h = hashlib.sha256()
with open(submission_dst, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(chunk)
sha = h.hexdigest()

meta = {
    "superadd_commit": COMMIT,
    "submission_sha256": sha,
    "categories": CATEGORIES,
    "paper_metric": "pixel standardized partial AU-ROC@max_fpr=0.05 + thresholded SegF1",
    "research_metric": "AU-PRO@FPR<=0.05",
}

(save_dir / "reproducibility.json").write_text(
    json.dumps(meta, indent=2)
)

print("SAVED:", save_dir)
print("UPLOAD THIS FILE:", submission_dst)
print("SHA256:", sha)

NameError: name 'submission' is not defined

## Kết quả nào dùng cho mục đích nào?

- `outputs_public/scores_test_public.csv` → **đối chiếu trực tiếp Table 1 SuperADD**.
- `outputs_public/aupro_005_test_public.csv` → **AU-PRO\(_{0.05}\)** cho nghiên cứu của bạn; paper SuperADD không báo cột này.
- `submissions/submission_*.tar.gz` → file nộp lên MVTec AD 2/VAND server cho **TESTpriv + TESTpriv,mix**.
- Không nộp `test_public` để lấy private score.

In [ ]:
# ============================================================
# CELL 9 — BACKUP OUTPUT CỦA CATEGORY "can" LÊN GOOGLE DRIVE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image

import hashlib
import json
import shutil
import tifffile
import numpy as np


CATEGORY = "fruit_jelly"

PRIVATE_SPLITS = [
    "test_private",
    "test_private_mixed",
]


# ============================================================
# 1. KIỂM TRA OUTPUT
# ============================================================

counts = {}

for split in PRIVATE_SPLITS:

    tiff_dir = (
        SUBMIT_OUT
        / "anomaly_images"
        / CATEGORY
        / split
    )

    png_dir = (
        SUBMIT_OUT
        / "anomaly_images_thresholded"
        / CATEGORY
        / split
    )

    assert tiff_dir.is_dir(), tiff_dir
    assert png_dir.is_dir(), png_dir

    tiffs = sorted(tiff_dir.glob("*.tiff"))
    pngs = sorted(png_dir.glob("*.png"))

    assert len(tiffs) > 0, f"Không có TIFF trong {tiff_dir}"
    assert len(pngs) > 0, f"Không có PNG trong {png_dir}"

    # Kiểm tra format một sample
    a = np.asarray(
        tifffile.imread(tiffs[0])
    )

    b = np.asarray(
        Image.open(pngs[0])
    )

    assert (
        a.ndim == 2
        and
        a.dtype == np.float16
    ), (
        f"TIFF sai format: "
        f"shape={a.shape}, dtype={a.dtype}"
    )

    assert b.ndim == 2

    assert set(
        np.unique(b).tolist()
    ).issubset({0, 255})

    counts[split] = {
        "tiff": len(tiffs),
        "png": len(pngs),
    }

    print(
        f"{split}: "
        f"{len(tiffs)} TIFF | "
        f"{len(pngs)} PNG"
    )


# ============================================================
# 2. TẠO THƯ MỤC BACKUP TRÊN DRIVE
# ============================================================

run_id = datetime.now(
    timezone.utc
).strftime(
    "%Y%m%dT%H%M%SZ"
)

save_dir = (
    DRIVE_OUT
    / run_id
)

save_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. COPY TIFF + PNG CỦA CAN
# ============================================================

for split in PRIVATE_SPLITS:

    # TIFF anomaly maps
    src_tiff = (
        SUBMIT_OUT
        / "anomaly_images"
        / CATEGORY
        / split
    )

    dst_tiff = (
        save_dir
        / "outputs"
        / "anomaly_images"
        / CATEGORY
        / split
    )

    shutil.copytree(
        src_tiff,
        dst_tiff,
        dirs_exist_ok=True
    )

    # PNG thresholded maps
    src_png = (
        SUBMIT_OUT
        / "anomaly_images_thresholded"
        / CATEGORY
        / split
    )

    dst_png = (
        save_dir
        / "outputs"
        / "anomaly_images_thresholded"
        / CATEGORY
        / split
    )

    shutil.copytree(
        src_png,
        dst_png,
        dirs_exist_ok=True
    )


# ============================================================
# 4. COPY KẾT QUẢ TEST_PUBLIC NẾU CÓ
# ============================================================

metric_files = [
    PUBLIC_OUT / "scores_test_public.csv",
    PUBLIC_OUT / "paper_comparison_test_public.csv",
    PUBLIC_OUT / "aupro_005_test_public.csv",
]

metric_dir = (
    save_dir
    / "metrics"
)

metric_dir.mkdir(
    parents=True,
    exist_ok=True
)

for p in metric_files:

    if p.is_file():

        shutil.copy2(
            p,
            metric_dir / p.name
        )


# ============================================================
# 5. LƯU METADATA
# ============================================================

meta = {
    "superadd_commit": COMMIT,
    "categories": [CATEGORY],
    "splits": PRIVATE_SPLITS,
    "counts": counts,
    "tiff_format": "float16, single-channel",
    "thresholded_png": "uint8 {0,255}",
}

(
    save_dir
    / "reproducibility.json"
).write_text(
    json.dumps(
        meta,
        indent=2
    )
)


# ============================================================
# 6. KIỂM TRA BACKUP
# ============================================================

print("\nBACKUP COMPLETE")
print("Saved to:", save_dir)

for split in PRIVATE_SPLITS:

    saved_tiffs = list(
        (
            save_dir
            / "outputs"
            / "anomaly_images"
            / CATEGORY
            / split
        ).glob("*.tiff")
    )

    saved_pngs = list(
        (
            save_dir
            / "outputs"
            / "anomaly_images_thresholded"
            / CATEGORY
            / split
        ).glob("*.png")
    )

    print(
        f"{split}: "
        f"{len(saved_tiffs)} TIFF | "
        f"{len(saved_pngs)} PNG"
    )

test_private: 321 TIFF | 321 PNG
test_private_mixed: 321 TIFF | 321 PNG

BACKUP COMPLETE
Saved to: /content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9/superadd_results_exact/20260910T181308Z
test_private: 321 TIFF | 321 PNG
test_private_mixed: 321 TIFF | 321 PNG
